## Executive Summary

**Purpose:** Describe any stator winding as coils placed at named positions in the slots, so that the inductance calculations (`emachines.inductance`) can tell where each phase sits inside a slot.

**What it does:** `WindingPattern` holds the signed turns of every phase at every position of every slot, `slot_turns[phase, slot, position]`, plus the coils that produced them. Builders make the common windings (star-of-slots, integer-slot distributed, tooth-coil concentrated) or take your own slot table or coil list. Every pattern can be checked for balance, gives its winding factors and draws itself as a developed slot diagram.

**Why a new object:** `build_coil_matrix` says only *top* or *bottom* layer, and a single layer has no position at all. Position-dependent slot leakage, side-by-side tooth coils and multi-layer (hairpin) windings need the position of every coil side by name.

## How It Fits Into the Motor Design Workflow

**Upstream:** `emachines.winding.sos` (star of slots, `build_coil_matrix`) and `design_windings`, which pick a balanced layout for (Q, P, m).

**Downstream:** `emachines.inductance` (stator geometry now; air-gap and slot-leakage inductance in the next steps). Every inductance part is computed from `slot_turns`.

## Conventions

- Slots are numbered 0 … Q − 1 in the direction of increasing angle, drawn left to right. Tooth *i* sits between slot *i* and slot *i* + 1.
- Phases are 0-based in code (0 = A) and lettered in text and plots.
- A coil side carries +turns where current flows **out of** the page and −turns where it flows **into** the page, for a positive phase current.
- Positions inside a slot depend on the layout:

| Layout | Positions | Meaning |
|---|---|---|
| `single` | `full` | one coil side fills the slot |
| `over_under`, 2 layers | `top`, `bottom` | `top` is on the air-gap side |
| `over_under`, n > 2 layers | `1` … `n` | `1` is on the air-gap side (hairpin and bar windings) |
| `side_by_side` | `left`, `right` | two halves across the slot width (tooth coils) |

The slot geometry (`emachines.inductance.SlotGeometry.regions`) maps each name to a region of the slot cross-section.

In [ ]:
#| default_exp winding.pattern

In [ ]:
#| export
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np

from emachines.winding.sos import (_PHASE_NAMES, _assign_slot_to_phase, _resolve_layout,
                                   build_coil_matrix, build_star_of_slots, check_winding)

In [ ]:
#| hide
import matplotlib.pyplot as plt

---

## Slot positions

In [ ]:
#| export
LAYOUTS = ("single", "over_under", "side_by_side")


def slot_positions(layout: str, n: int = 2) -> tuple[str, ...]:
    """
    Names of the conductor positions in a slot for a given layout.

    Parameters
    ----------
    layout : str  'single', 'over_under' or 'side_by_side' (see ``LAYOUTS``)
    n      : int  Number of radial layers for 'over_under' (default 2); ignored otherwise

    Returns
    -------
    tuple of str
        'single' -> ('full',); 'over_under' with n = 2 -> ('top', 'bottom');
        'over_under' with n > 2 -> ('1', ..., 'n'), '1' on the air-gap side;
        'side_by_side' -> ('left', 'right').
    """
    if layout == "single":
        return ("full",)
    if layout == "over_under":
        if n < 2:
            raise ValueError(f"an over-under layout needs n >= 2 layers, got n={n}")
        return ("top", "bottom") if n == 2 else tuple(str(k) for k in range(1, n + 1))
    if layout == "side_by_side":
        return ("left", "right")
    raise ValueError(f"layout must be one of {LAYOUTS}, got {layout!r}")

In [ ]:
# Examples
assert slot_positions("single") == ("full",)
assert slot_positions("over_under") == ("top", "bottom")
assert slot_positions("over_under", 4) == ("1", "2", "3", "4")
assert slot_positions("side_by_side") == ("left", "right")

---

## `Coil`

In [ ]:
#| export
@dataclass(frozen=True)
class Coil:
    """
    One coil of one phase, with a side in each of two slot positions.

    For a positive phase current, current flows out of the page in the `out`
    side and into the page in the `in` side.

    Attributes
    ----------
    phase    : int   Phase index, 0-based (0 = A)
    out_slot : int   Slot of the out side, 0 ... Q-1
    out_pos  : str   Position of the out side in its slot (see ``slot_positions``)
    in_slot  : int   Slot of the in side
    in_pos   : str   Position of the in side
    turns    : int   Turns in the coil (default 1)
    """
    phase: int
    out_slot: int
    out_pos: str
    in_slot: int
    in_pos: str
    turns: int = 1

---

## `WindingPattern`

The pattern stores `slot_turns`, shape (m, Q, number of positions). Entry `[k, i, r]` is the signed number of turns of phase *k* at position *r* of slot *i*, for one parallel path. With *a* parallel paths in each phase, every path carries 1/*a* of the phase current, so the turns that link flux per ampere of phase current are `slot_turns / a`. That is `effective_turns()`, and it is what the inductance calculations use.

The balance check works on `slot_turns` directly. Phase *k* has the EMF phasor

$$E_k = \sum_i \sum_r n_{k,i,r}\, e^{\,j p\, 2\pi i/Q},$$

and its winding factor at harmonic ν is $k_{w\nu} = |\sum n\, e^{j\nu 2\pi i/Q}| / \sum |n|$. The winding is **buildable** if each phase has as many out turns as in turns, and **balanced** if the m phasors are equal in size and 360°/m apart (in either sequence). This is the same test as `check_winding`, extended to any number of positions and to turns.

In [ ]:
#| export
@dataclass(frozen=True)
class WindingPattern:
    """
    A stator winding as signed turns per phase, slot and slot position.

    Build one with a classmethod (``from_star_of_slots``, ``distributed``,
    ``concentrated``, ``from_slot_table`` or ``custom``) rather than directly.

    Attributes
    ----------
    Q, P, m        : int              Slots, poles, phases
    layout         : str              'single', 'over_under' or 'side_by_side'
    positions      : tuple of str     Position names, as from ``slot_positions``
    slot_turns     : np.ndarray       Shape (m, Q, len(positions)), signed turns per path
    coils          : tuple | None     The coils, or None if built from a slot table
    parallel_paths : int              Parallel paths per phase, a (default 1)
    name           : str              Short description
    """
    Q: int
    P: int
    m: int
    layout: str
    positions: tuple
    slot_turns: np.ndarray = field(repr=False, compare=False)
    coils: tuple | None = field(default=None, repr=False, compare=False)
    parallel_paths: int = 1
    name: str = ""

    def __post_init__(self):
        if self.P < 2 or self.P % 2:
            raise ValueError(f"P must be an even number of poles >= 2, got P={self.P}")
        if self.layout not in LAYOUTS:
            raise ValueError(f"layout must be one of {LAYOUTS}, got {self.layout!r}")
        if self.parallel_paths < 1:
            raise ValueError(f"parallel_paths must be >= 1, got {self.parallel_paths}")
        n = np.asarray(self.slot_turns, dtype=float)
        if n.shape != (self.m, self.Q, len(self.positions)):
            raise ValueError(f"slot_turns must have shape (m, Q, positions) = "
                             f"{(self.m, self.Q, len(self.positions))}, got {n.shape}")
        object.__setattr__(self, "slot_turns", n)

    @property
    def p(self) -> int:
        """Pole pairs."""
        return self.P // 2

    def effective_turns(self) -> np.ndarray:
        """Turns per ampere of phase current, ``slot_turns / parallel_paths``."""
        return self.slot_turns / self.parallel_paths

    def series_turns(self) -> np.ndarray:
        """Series turns per phase, N_s = (sum of |turns| / 2) / a, one value per phase."""
        return np.abs(self.slot_turns).sum(axis=(1, 2)) / 2 / self.parallel_paths

    def _phasors(self, nu: int) -> np.ndarray:
        e = np.exp(1j * nu * 2.0 * np.pi * np.arange(self.Q) / self.Q)
        return np.einsum("kir,i->k", self.slot_turns, e)

    def winding_factor(self, nu: int | None = None) -> np.ndarray:
        """|k_w| of each phase at mechanical harmonic order nu (default nu = p)."""
        nu = self.p if nu is None else nu
        total = np.abs(self.slot_turns).sum(axis=(1, 2))
        return np.abs(self._phasors(nu)) / np.where(total > 0, total, 1.0)

    def check(self) -> dict:
        """
        Check that the pattern is buildable and balanced.

        Returns
        -------
        dict with keys:
            buildable    : bool   Each phase has as many out turns as in turns
            balanced     : bool   Phase EMFs at nu = p equal in size and 360°/m apart
            kw1          : float  Fundamental winding factor of phase A
            sequence     : str    'positive' (A, B, C ... lag by 360°/m), 'negative', or ''
            phase_angles : list | None
                                  Angle of each phase EMF relative to phase A, degrees
            reason       : str    Why the check fails, or '' if it passes
        """
        n = self.slot_turns
        out_t = np.where(n > 0, n, 0).sum(axis=(1, 2))
        in_t = -np.where(n < 0, n, 0).sum(axis=(1, 2))
        reason = ""
        buildable = bool(np.allclose(out_t, in_t))
        if not buildable:
            k = int(np.argmax(~np.isclose(out_t, in_t)))
            reason = (f"phase {_PHASE_NAMES[k]} has {out_t[k]:g} out turns and {in_t[k]:g} in turns, "
                      f"so it can't be wound into coils")
        E = self._phasors(self.p)
        mag = np.abs(E)
        total = out_t + in_t
        sequence = ""
        if mag.min() < 1e-9:
            balanced, phase_angles = False, None
            reason = reason or "a phase EMF at the working harmonic is zero"
        else:
            steps = np.angle(E[1:] / E[:-1]) % (2 * np.pi) if self.m > 1 else np.array([])
            same = (bool(np.allclose(total, total[0])) and bool(np.allclose(mag, mag[0], rtol=1e-9)))
            if np.allclose(steps, 2 * np.pi / self.m, atol=1e-9):
                sequence = "positive"
            elif np.allclose(steps, 2 * np.pi - 2 * np.pi / self.m, atol=1e-9):
                sequence = "negative"
            balanced = same and bool(sequence)
            phase_angles = [float(a) % 360.0 for a in np.round(np.degrees(np.angle(E / E[0])), 6)]
            if not balanced:
                reason = reason or "phases are not equal in size and 360°/m apart"
        kw1 = float(mag[0] / total[0]) if total[0] else 0.0
        return dict(buildable=buildable, balanced=balanced, kw1=kw1, sequence=sequence,
                    phase_angles=phase_angles, reason=reason)

    def describe(self) -> str:
        """One-line summary."""
        r = self.check()
        ok = "balanced" if r["balanced"] else f"NOT balanced ({r['reason']})"
        return (f"{self.name or 'winding'}: {self.Q}s/{self.P}p, m={self.m}, {self.layout} "
                f"{self.positions}, N_s = {self.series_turns()[0]:g}, kw1 = {r['kw1']:.4f}, {ok}")

    def plot(self, ax=None):
        """Draw the developed slot diagram: each position coloured by phase, marked + (out) or - (in)."""
        import matplotlib.pyplot as plt
        from matplotlib.patches import Rectangle
        if ax is None:
            _, ax = plt.subplots(figsize=(max(6.0, 0.42 * self.Q), 2.6))
        colors = plt.get_cmap("tab10").colors
        npos = len(self.positions)
        for i in range(self.Q):
            ax.add_patch(Rectangle((i, 0), 0.8, 1, fill=False, lw=1.0, ec="#4a5160"))
            for r in range(npos):
                if self.layout == "side_by_side":
                    x0, y0, w, h = i + 0.4 * r, 0.0, 0.4, 1.0
                else:
                    x0, y0, w, h = i, 1.0 - (r + 1) / npos, 0.8, 1.0 / npos
                for k in range(self.m):
                    t = self.slot_turns[k, i, r]
                    if t == 0:
                        continue
                    ax.add_patch(Rectangle((x0 + 0.04, y0 + 0.04), w - 0.08, h - 0.08,
                                           fc=colors[k % 10], ec="#222", lw=0.6, alpha=0.85))
                    ax.text(x0 + w / 2, y0 + h / 2, f"{'+' if t > 0 else '−'}{_PHASE_NAMES[k]}",
                            ha="center", va="center", fontsize=7)
            ax.text(i + 0.4, -0.18, str(i), ha="center", va="top", fontsize=7)
        ax.set_xlim(-0.3, self.Q + 0.1)
        ax.set_ylim(-0.45, 1.25)
        ax.set_axis_off()
        ax.set_title(self.name or "winding pattern", fontsize=10)
        ax.text(-0.25, 1.08, "air gap side", fontsize=7, color="#4a5160")
        return ax

    # ------------------------------------------------------------------ builders
    @classmethod
    def custom(cls, Q: int, P: int, coils, m: int = 3, layout: str = "over_under", n_layers: int = 2,
               parallel_paths: int = 1, name: str = "custom") -> "WindingPattern":
        """
        A pattern from an explicit list of ``Coil`` objects.

        Parameters
        ----------
        Q, P, m        : int   Slots, poles, phases
        coils          : iterable of Coil
        layout         : str   Slot layout (default 'over_under')
        n_layers       : int   Radial layers for 'over_under' (default 2)
        parallel_paths : int   Parallel paths per phase (default 1)
        name           : str   Short description
        """
        positions = slot_positions(layout, n_layers)
        index = {p: r for r, p in enumerate(positions)}
        coils = tuple(coils)
        n = np.zeros((m, Q, len(positions)))
        for c in coils:
            if not 0 <= c.phase < m:
                raise ValueError(f"coil phase must be 0 ... {m - 1}, got {c.phase}")
            for slot, pos, sign in ((c.out_slot, c.out_pos, 1), (c.in_slot, c.in_pos, -1)):
                if not 0 <= slot < Q:
                    raise ValueError(f"coil slot must be 0 ... {Q - 1}, got {slot}")
                if pos not in index:
                    raise ValueError(f"position {pos!r} is not one of {positions} for layout {layout!r}")
                n[c.phase, slot, index[pos]] += sign * c.turns
        return cls(Q, P, m, layout, positions, n, coils, parallel_paths, name)

    @classmethod
    def from_slot_table(cls, table, P: int, m: int = 3, turns: int = 1, layout: str | None = None,
                        parallel_paths: int = 1, name: str = "slot table") -> "WindingPattern":
        """
        A pattern from a table of signed phase codes, one row per position.

        Parameters
        ----------
        table          : array-like, shape (positions, Q)
                         +k = phase k out of the page, -k = into the page, 0 = empty
                         (phases 1-based, as in ``build_coil_matrix``)
        P, m           : int   Poles, phases
        turns          : int   Turns in each coil side (default 1)
        layout         : str | None
                         None -> 'single' for one row, 'over_under' for more rows.
                         'side_by_side' needs two rows (left, right).
        parallel_paths : int   Parallel paths per phase (default 1)
        name           : str   Short description

        The coils are not known from a table, so ``coils`` is None.
        """
        table = np.atleast_2d(np.asarray(table, dtype=int))
        rows, Q = table.shape
        if layout is None:
            layout = "single" if rows == 1 else "over_under"
        positions = slot_positions(layout, rows)
        if len(positions) != rows:
            raise ValueError(f"layout {layout!r} has {len(positions)} positions but the table has {rows} rows")
        if np.abs(table).max() > m:
            raise ValueError(f"phase codes must be within ±{m}")
        n = np.zeros((m, Q, rows))
        for r in range(rows):
            for i in range(Q):
                c = int(table[r, i])
                if c:
                    n[abs(c) - 1, i, r] += np.sign(c) * turns
        return cls(Q, P, m, layout, positions, n, None, parallel_paths, name)

    @classmethod
    def from_star_of_slots(cls, Q: int, P: int, m: int = 3, layers: int | None = None,
                           w: int | None = None, turns: int = 1, parallel_paths: int = 1) -> "WindingPattern":
        """
        The star-of-slots winding of ``build_coil_matrix`` as a pattern.

        Parameters
        ----------
        Q, P, m        : int          Slots, poles, phases
        layers         : int | None   1 or 2; None -> the best buildable, balanced layout
        w              : int | None   Double-layer coil span in slots; None -> the best span
        turns          : int          Turns per coil (default 1)
        parallel_paths : int          Parallel paths per phase (default 1)

        Double layer: each coil runs from the top of slot i to the bottom of slot i + w.
        Single layer: each out side is paired with the nearest free in side of the same
        phase, counting forward; the pairing sets the coils but not ``slot_turns``.
        """
        layers, w = _resolve_layout(Q, P, m, layers, w)
        matrix = build_coil_matrix(Q, P, m, layers, w)
        coils = []
        if layers == 2:
            assert w is not None
            for i in range(Q):
                code, j = int(matrix[0, i]), (i + w) % Q
                k = abs(code) - 1
                if code > 0:
                    coils.append(Coil(k, i, "top", j, "bottom", turns))
                else:
                    coils.append(Coil(k, j, "bottom", i, "top", turns))
            return cls.custom(Q, P, coils, m, "over_under", 2, parallel_paths,
                              name=f"star of slots, double layer, w={w}")
        for k in range(m):
            outs = [i for i in range(Q) if matrix[0, i] == k + 1]
            ins = [i for i in range(Q) if matrix[0, i] == -(k + 1)]
            for o in outs:
                j = min(ins, key=lambda s: (s - o) % Q)
                ins.remove(j)
                coils.append(Coil(k, o, "full", j, "full", turns))
        return cls.custom(Q, P, coils, m, "single", parallel_paths=parallel_paths,
                          name="star of slots, single layer")

    @classmethod
    def distributed(cls, Q: int, P: int, m: int = 3, layers: int = 2, pitch: int | None = None,
                    turns: int = 1, parallel_paths: int = 1) -> "WindingPattern":
        """
        An integer-slot lap winding (q = Q/(P m) a whole number), full or short pitch.

        Parameters
        ----------
        Q, P, m        : int          Slots, poles, phases
        layers         : int          1 or 2 (default 2)
        pitch          : int | None   Double-layer coil pitch in slots; None -> full pitch Q/P.
                                      Not allowed for a single layer, whose pitch is fixed.
        turns          : int          Turns per coil (default 1)
        parallel_paths : int          Parallel paths per phase (default 1)
        """
        if Q % (P * m):
            raise ValueError(f"{Q}s/{P}p/m={m} is not an integer-slot winding (q = {Q / (P * m):g}); "
                             "use from_star_of_slots or concentrated")
        if layers == 1 and pitch is not None:
            raise ValueError("a single-layer winding has a fixed pitch; leave pitch=None")
        w = None if layers == 1 else (Q // P if pitch is None else pitch)
        pat = cls.from_star_of_slots(Q, P, m, layers, w, turns, parallel_paths)
        q = Q // (P * m)
        kind = "single layer" if layers == 1 else f"double layer, pitch {w}/{Q // P}"
        return _renamed(pat, f"distributed, q={q}, {kind}")

    @classmethod
    def concentrated(cls, Q: int, P: int, m: int = 3, all_teeth: bool = True,
                     layout: str = "side_by_side", turns: int = 1, parallel_paths: int = 1) -> "WindingPattern":
        """
        A tooth-coil (concentrated, non-overlapping) winding.

        Parameters
        ----------
        Q, P, m        : int    Slots, poles, phases
        all_teeth      : bool   True: a coil on every tooth (double layer).
                                False: a coil on every other tooth (single layer, Q even).
        layout         : str    For all_teeth: 'side_by_side' (default) or 'over_under'.
                                Ignored for alternate teeth, where each slot holds one coil side.
        turns          : int    Turns per coil (default 1)
        parallel_paths : int    Parallel paths per phase (default 1)

        The coil on tooth i has its sides in slot i and slot i + 1. Side by side, that
        is the right half of slot i and the left half of slot i + 1; over-under, the top
        of slot i and the bottom of slot i + 1. Phases follow the star of slots, and the
        result must pass ``check``.
        """
        if all_teeth:
            if layout not in ("side_by_side", "over_under"):
                raise ValueError("all-teeth windings use layout 'side_by_side' or 'over_under'")
            matrix = build_coil_matrix(Q, P, m, layers=2, w=1)
            r = check_winding(matrix, P, m)
            if not (r["buildable"] and r["balanced"]):
                raise ValueError(f"no balanced all-teeth winding for {Q}s/{P}p (m={m}): {r['reason']}")
            a, b = ("right", "left") if layout == "side_by_side" else ("top", "bottom")
            coils = []
            for i in range(Q):
                code, j = int(matrix[0, i]), (i + 1) % Q
                k = abs(code) - 1
                coils.append(Coil(k, i, a, j, b, turns) if code > 0 else Coil(k, j, b, i, a, turns))
            pat = cls.custom(Q, P, coils, m, layout, 2, parallel_paths)
            return _renamed(pat, f"concentrated, all teeth, {layout.replace('_', '-')}")
        if Q % 2:
            raise ValueError(f"alternate-teeth windings need an even slot count, got Q={Q}")
        angles = build_star_of_slots(Q, P)
        coils = []
        for i in range(0, Q, 2):
            code = _assign_slot_to_phase(angles[i], m)
            k, j = abs(code) - 1, (i + 1) % Q
            coils.append(Coil(k, i, "full", j, "full", turns) if code > 0 else Coil(k, j, "full", i, "full", turns))
        pat = cls.custom(Q, P, coils, m, "single", parallel_paths=parallel_paths,
                         name="concentrated, alternate teeth")
        r = pat.check()
        if not r["balanced"]:
            raise ValueError(f"no balanced alternate-teeth winding for {Q}s/{P}p (m={m}): {r['reason']}")
        return pat


def _renamed(pat: WindingPattern, name: str) -> WindingPattern:
    from dataclasses import replace
    return replace(pat, name=name)

---

## Examples

A 36-slot, 4-pole, short-pitched double-layer winding (pitch 7 of 9 slots), with 10 turns per coil and 2 parallel paths:

In [ ]:
pat = WindingPattern.distributed(36, 4, layers=2, pitch=7, turns=10, parallel_paths=2)
print(pat.describe())
assert pat.check()["balanced"] and np.isclose(pat.check()["kw1"], 0.9019, atol=1e-4)
assert np.allclose(pat.series_turns(), 36 * 10 / 3 / 2)        # 12 coils of 10 turns per phase, 2 paths
pat.plot();

The short pitch puts two phases in some slots, one in the top layer and one in the bottom. These are the slots where the cross-slot mutual inductance comes from:

In [ ]:
shared = [i for i in range(pat.Q) if (np.abs(pat.slot_turns[:, i, :]).sum(axis=1) > 0).sum() == 2]
print(f"slots shared by two phases: {shared}")
assert len(shared) == 24

Tooth-coil windings for 12 slots and 10 poles, side by side on every tooth and single layer on alternate teeth:

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(7, 4.8))
for ax, p in zip(axes, [WindingPattern.concentrated(12, 10),
                        WindingPattern.concentrated(12, 10, all_teeth=False)]):
    print(p.describe())
    p.plot(ax=ax)
plt.tight_layout()

Your own winding, from a slot table (rows are positions, top first) or from a coil list:

In [ ]:
table = [[1, 1, -3, -3, 2, 2, -1, -1, 3, 3, -2, -2],      # top layer
         [1, -3, -3, 2, 2, -1, -1, 3, 3, -2, -2, 1]]      # bottom layer
own = WindingPattern.from_slot_table(table, P=2, turns=5)
print(own.describe())

coils = [Coil(0, 0, "top", 5, "bottom", 8), Coil(1, 4, "top", 9, "bottom", 8), Coil(2, 8, "top", 1, "bottom", 8)]
print(WindingPattern.custom(12, 2, coils).describe())

---

## Tests

In [ ]:
#| hide
import itertools
from emachines.winding.sos import build_coil_matrix, check_winding, winding_factor_sos
from emachines.winding.design import design_windings

# star-of-slots patterns agree with check_winding and winding_factor_sos, every candidate
for Q, P in [(12, 10), (9, 8), (24, 4), (36, 8), (15, 8), (36, 4), (48, 8), (18, 4)]:
    for c in design_windings(Q, P):
        pat = WindingPattern.from_star_of_slots(Q, P, layers=c.layers, w=c.w)
        r = pat.check()
        assert r["buildable"] and r["balanced"], (Q, P, c.layers, c.w, r)
        assert np.isclose(r["kw1"], c.kw1), (Q, P, c.layers, c.w)
        for nu in range(1, 20):
            assert np.isclose(pat.winding_factor(nu)[0], abs(winding_factor_sos(nu, Q, P, 3, c.layers, c.w))), (Q, P, nu)
        # the table route and the coil route give the same slot turns
        tab = WindingPattern.from_slot_table(build_coil_matrix(Q, P, 3, c.layers, c.w), P)
        assert np.array_equal(tab.slot_turns, pat.slot_turns)
        # every coil has one out and one in side, so turns cancel per phase
        assert np.allclose(pat.slot_turns.sum(axis=(1, 2)), 0)

# turns and parallel paths scale slot turns and series turns
p1 = WindingPattern.distributed(24, 4, pitch=5)
p2 = WindingPattern.distributed(24, 4, pitch=5, turns=7, parallel_paths=2)
assert np.array_equal(p2.slot_turns, 7 * p1.slot_turns)
assert np.allclose(p2.effective_turns(), 3.5 * p1.slot_turns)
assert np.allclose(p1.series_turns(), 8) and np.allclose(p2.series_turns(), 28)

# full-pitch double layer: every slot has one phase in both layers; short pitch shares slots
full = WindingPattern.distributed(24, 4)
assert all((np.abs(full.slot_turns[:, i, :]).sum(axis=1) > 0).sum() == 1 for i in range(24))
assert np.isclose(full.check()["kw1"], 0.9659, atol=1e-4)
single = WindingPattern.distributed(24, 4, layers=1)
assert single.positions == ("full",) and np.isclose(single.check()["kw1"], 0.9659, atol=1e-4)
assert all(c.in_slot == (c.out_slot + 6) % 24 for c in single.coils)   # full-pitch coils

# tooth coils: published kw1 for 12s/10p, 9s/8p, 12s/8p
for (Q, P), kw in {(12, 10): 0.9330, (9, 8): 0.9452, (12, 8): 0.8660}.items():
    for lay in ("side_by_side", "over_under"):
        t = WindingPattern.concentrated(Q, P, layout=lay)
        assert t.check()["balanced"] and np.isclose(t.check()["kw1"], kw, atol=1e-4), (Q, P, lay)
        assert all((c.in_slot - c.out_slot) % Q in (1, Q - 1) for c in t.coils)
alt = WindingPattern.concentrated(12, 10, all_teeth=False)
assert alt.check()["balanced"] and np.isclose(alt.check()["kw1"], 0.9659, atol=1e-4)
assert len(alt.coils) == 6 and all(np.abs(alt.slot_turns[:, i, 0]).sum() == 1 for i in range(12))
sbs = WindingPattern.concentrated(12, 10)
assert np.array_equal(np.abs(sbs.slot_turns).sum(axis=0), np.ones((12, 2)))    # one side in each half

# custom coil list: position bookkeeping and validation
c = WindingPattern.custom(6, 2, [Coil(0, 0, "top", 3, "bottom", 4)])
assert c.slot_turns[0, 0, 0] == 4 and c.slot_turns[0, 3, 1] == -4 and np.count_nonzero(c.slot_turns) == 2
for bad in [dict(coils=[Coil(3, 0, "top", 3, "bottom")]),
            dict(coils=[Coil(0, 6, "top", 3, "bottom")]),
            dict(coils=[Coil(0, 0, "left", 3, "bottom")])]:
    try:
        WindingPattern.custom(6, 2, **bad); raise AssertionError(bad)
    except ValueError:
        pass
assert WindingPattern.custom(12, 4, [Coil(0, 0, "1", 3, "4")], layout="over_under", n_layers=4).positions == ("1", "2", "3", "4")

# unbalanced and unbuildable patterns are reported, not hidden
r = WindingPattern.custom(6, 2, [Coil(0, 0, "top", 3, "bottom")]).check()
assert r["buildable"] and not r["balanced"]
r = WindingPattern.from_slot_table([[1, 0, 0, 0, 0, 0]], P=2).check()
assert not r["buildable"] and "phase A" in r["reason"]

# negative sequence still counts as balanced
neg = WindingPattern(pat.Q, pat.P, 3, pat.layout, pat.positions, pat.slot_turns[[0, 2, 1]])
assert neg.check()["balanced"] and neg.check()["sequence"] == "negative"

# argument errors
for f in [lambda: WindingPattern.distributed(12, 10), lambda: WindingPattern.distributed(24, 4, layers=1, pitch=5),
          lambda: WindingPattern.concentrated(9, 8, all_teeth=False), lambda: slot_positions("diagonal"),
          lambda: slot_positions("over_under", 1), lambda: WindingPattern.from_slot_table([[1, 2]], P=2, layout="side_by_side"),
          lambda: WindingPattern(6, 3, 3, "single", ("full",), np.zeros((3, 6, 1)))]:
    try:
        f(); raise AssertionError("expected ValueError")
    except ValueError:
        pass

## References

- Bianchi, N., Dai Prè, M. (2006). Use of the star of slots in designing fractional-slot single-layer synchronous motors. *IEE Proc. Electric Power Applications* 153(3) — star of slots, single-layer tooth-coil windings.
- Pyrhönen, J., Jokinen, T., Hrabovcová, V. (2014). *Design of Rotating Electrical Machines* (2nd ed.). Wiley — Chapter 2, windings.
- Sudhoff, S.D. *Power Magnetic Devices* (2nd ed.), §8.1–8.5 — conductor distribution by slot, used here per slot position.